In [ ]:
# rende config.py (in notebooks/) importabile anche dai notebook in sottocartelle
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import logging
from pathlib import Path
import albedo_functions as af
import itertools
from dask.distributed import Client, LocalCluster
import concurrent.futures
from scipy.stats import pearsonr, ttest_rel

# --- Logging ---
log_file = "process_log.txt"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file, mode='w'),
    ]
)

# --- Paths and Params ---
DATA_PATH = POST_DATA
OUTPUT_PATH = FIG_DIR
OBS_PATH = WORK_DIR

exp_sens = "a52o"
exp_ctrl = "a1ua"
var = "tas"
realm = "Amon"
seasons = ['DJF', 'MAM', 'JJA', 'SON']

# --- Utility Functions ---
def assign_djf_year(time_array):
    timestamps = pd.to_datetime(time_array)
    return [t.year + 1 if t.month == 12 else t.year for t in timestamps]

def normalize_time_and_group(df, time_col='time'):
    df[time_col] = pd.to_datetime(df[time_col])
    df['year'] = pd.Series(assign_djf_year(df[time_col])).astype(int)
    return df

def adjust_time_for_season(ds, season):
    ds['time'] = pd.to_datetime(ds['time'].values).normalize().to_period('M').start_time
    return ds

def load_dataset(exp, var, realm, lead, season, data_path):
    path = data_path / f"{exp}/1x1/{var}/{exp}_{var}_{realm}_EC-Earth3_dcppA-hindcast_lead_{lead}_1x1_ensemble_m{season}_rad.nc"
    ds = xr.open_dataset(path, chunks={'time': -1})
    
    #if season == 'DJF':
    #    ds['time'] = [pd.Timestamp(t).replace(month=12) for t in ds['time'].values]
    #elif season == 'MAM':
    #    ds['time'] = [pd.Timestamp(t).replace(month=3) for t in ds['time'].values]
    #    ds['time'] = pd.to_datetime(ds['time']) + pd.DateOffset(years=1)
    #elif season == 'JJA':
    #    ds['time'] = [pd.Timestamp(t).replace(month=6) for t in ds['time'].values]
    #    ds['time'] = pd.to_datetime(ds['time']) + pd.DateOffset(years=1)
    #elif season == 'SON':
    #    ds['time'] = [pd.Timestamp(t).replace(month=9) for t in ds['time'].values]
        
    if var == 'snd':
        ds['lon'] = ds['lon'] % 360
        ds = ds.sortby('lon')
    ds = adjust_time_for_season(ds, season)
    return ds

def prepare_df(ds, var, domain_selection_func):
    ds_dom = domain_selection_func(ds, -90, 90, 0, 360)
    df = (ds_dom[var] - ds_dom[var].sel(time=slice('1998','2014')).mean("time")).to_dataframe().reset_index()
    df = normalize_time_and_group(df)
    return df

def compute_metrics(df_model, df_obs, var):
    merged = pd.merge(df_model.groupby('year')[var].mean().reset_index(),
                      df_obs.groupby('year')[var].mean().reset_index(),
                      on='year', suffixes=('_model', '_obs'))
    rmse = ((merged[f'{var}_model'] - merged[f'{var}_obs']) ** 2).mean() ** 0.5
    corr, pval = pearsonr(merged[f'{var}_model'], merged[f'{var}_obs'])
    return rmse, corr, pval, merged[f'{var}_model'], merged[f'{var}_obs']

def plot_distribution(df_models, df_obs, var, season, lead, rmse_ctrl, corr_ctrl, pval_ctrl, rmse_sens, corr_sens, pval_sens, p_diff):
    logging.info(f"Plotting distribution for {var} - {season} - Lead {lead}")

    df_models = df_models.copy()
    df_obs = df_obs.copy()

    # Ensure 'year' is converted to integers
    df_models['year'] = pd.to_datetime(df_models['year'], format='%Y').dt.year
    df_obs['year'] = pd.to_datetime(df_obs['year'], format='%Y').dt.year

    fig, ax = plt.subplots(figsize=(18, 6))
    sns.boxplot(data=df_models, x='year', y=var, hue='model', showfliers=False, ax=ax)
    sns.pointplot(data=df_obs, x='year', y=var, ax=ax, color='Red', label='ERA5')
    ax.set_xlabel("Year")
    ax.set_ylabel(var)
    ax.set_title(f"Members' Distribution - {season} - Lead {lead}")
    ax.grid(True)
    ax.legend(loc='upper right')

    metrics_text = (
        f"Control\nRMSE: {rmse_ctrl:.3f}\nCorr: {corr_ctrl:.3f}\nP-val: {pval_ctrl:.3f}\n\n"
        f"Sensitive\nRMSE: {rmse_sens:.3f}\nCorr: {corr_sens:.3f}\nP-val: {pval_sens:.3f}\n\n"
        f"Δ Error p-val: {p_diff:.3f}"
    )
    fig.text(0.01, 0.98, metrics_text, fontsize=12,
             verticalalignment='top', bbox=dict(facecolor='white', edgecolor='black'))

    plt.tight_layout()
    plot_filename = OUTPUT_PATH / f"fig5_members_distribution_{var}_{season}_lead_{lead}_1999.png"
    plt.savefig(plot_filename)
    plt.close()
    logging.info(f"Saved plot: {plot_filename}")

def process_lead_season(y1, y2, season):
    lead = f"{y1}-{y2}"
    lead_number = y2 - y1 + 1
    try:
        dset_ctrl = load_dataset(exp_ctrl, var, realm, lead, season, DATA_PATH)
        dset_sens = load_dataset(exp_sens, var, realm, lead, season, DATA_PATH)

        obs_file_map = {
            'alb': f"GLASS_1x1_{lead_number}{season}.nc",
            'tas': f"ERA5_tas_1x1_{lead_number}{season}.nc",
            'sd': f"ERA5_sd_1x1_{lead_number}{season}.nc",
            'snd': f"ERA5_snd_1x1_{lead_number}{season}.nc"
        }
        obs_path = OBS_PATH / obs_file_map[var]
        obs = xr.open_dataset(obs_path, chunks={'time': -1})
        obs = adjust_time_for_season(obs, season)
        rename_map = {'alb': 'bb_albedo', 'tas': '2t'}
        if var in rename_map:
            obs = obs.rename({rename_map[var]: var})
        obs = obs.sel(time=slice(dset_ctrl.time[0], dset_ctrl.time[-1]))

        dset_ctrl = dset_ctrl.sel(time=slice('1999',None))
        dset_sens = dset_sens.sel(time=slice('1999',None))
        obs = obs.sel(time=slice('1999', None))

        
        df_ctrl = prepare_df(dset_ctrl, var, af.domain_selection)
        df_sens = prepare_df(dset_sens, var, af.domain_selection)
        df_obs = prepare_df(obs, var, af.domain_selection)

        df_ctrl['model'] = 'DCPP-CTRL'
        df_sens['model'] = 'DCPP-SENS'
        df_models = pd.concat([df_ctrl, df_sens], ignore_index=True)

        rmse_ctrl, corr_ctrl, pval_ctrl, ctrl_vals, obs_vals = compute_metrics(df_ctrl, df_obs, var)
        logging.info(df_sens)
        rmse_sens, corr_sens, pval_sens, sens_vals, _ = compute_metrics(df_sens, df_obs, var)
        t_stat, p_diff = ttest_rel(abs(ctrl_vals - obs_vals), abs(sens_vals - obs_vals))

        plot_distribution(df_models, df_obs, var, season, lead,
                          rmse_ctrl, corr_ctrl, pval_ctrl,
                          rmse_sens, corr_sens, pval_sens, p_diff)

    except Exception as e:
        logging.error(f"Failed to process {season} lead {lead}: {e}")

lead_pairs = [(i, j) for i in range(5) for j in range(i+1, 5)] + [(i, i) for i in range(5)]
lead_combinations = lead_pairs
all_jobs = [(y1, y2, season) for (y1, y2) in lead_combinations for season in seasons]

with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
    executor.map(lambda args: process_lead_season(*args), all_jobs)
